# Exploratory Data Analysis (EDA) with SQL

## Overview
In this notebook, we perform Exploratory Data Analysis (EDA) on the SpaceX Falcon 9 launch dataset using SQL queries (`sqlite3` and `ipython-sql` / `pandas`).

### Objectives
1. Understand the Spacex dataset schema and unique launch sites.
2. Filter records and aggregate payload masses by customer and booster version.
3. Analyze mission outcomes, first successful ground-pad landing dates, and drone-ship landing milestones.
4. Rank landing outcomes within specific date ranges.

In [1]:
import sqlite3
import pandas as pd

# Connect to SQLite database and load the SpaceX dataset
con = sqlite3.connect('my_data1.db')
cur = con.cursor()

# Load CSV into SQLite table (replace URL/path with local dataset if needed)
# df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv')
# df.to_sql('SPACEXTBL', con, if_exists='replace', index=False)
print('Database connection established and SPACEXTBL ready.')

Database connection established and SPACEXTBL ready.


### Task 1: Display the names of the unique launch sites in the space mission

In [2]:
query_1 = 'SELECT DISTINCT Launch_Site FROM SPACEXTBL;'
pd.read_sql_query(query_1, con)

 Launch_Site
 CCAFS LC-40
 VAFB SLC-4E
  KSC LC-39A
CCAFS SLC-40


### Task 2: Display 5 records where launch sites begin with the string `'CCA'`

In [3]:
query_2 = "SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5;"
pd.read_sql_query(query_2, con)

      Date Time (UTC) Booster_Version Launch_Site                                                       Payload  PAYLOAD_MASS__KG_     Orbit        Customer Mission_Outcome     Landing_Outcome
2010-06-04   18:45:00  F9 v1.0  B0003 CCAFS LC-40                          Dragon Spacecraft Qualification Unit                  0       LEO          SpaceX         Success Failure (parachute)
2010-12-08   15:43:00  F9 v1.0  B0004 CCAFS LC-40 Dragon demo flight C1, two CubeSats, barrel of Brouere cheese                  0 LEO (ISS) NASA (COTS) NRO         Success Failure (parachute)
2012-05-22   07:44:00  F9 v1.0  B0005 CCAFS LC-40                                         Dragon demo flight C2                525 LEO (ISS)     NASA (COTS)         Success          No attempt
2012-10-08   00:35:00  F9 v1.0  B0006 CCAFS LC-40                                                  SpaceX CRS-1                500 LEO (ISS)      NASA (CRS)         Success          No attempt
2013-03-01   15:10:00  F9 v1.0  B00

### Task 3: Display the total payload mass carried by boosters launched by NASA (CRS)

In [4]:
query_3 = """
SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_KG
FROM SPACEXTBL
WHERE Customer = 'NASA (CRS)';
"""
pd.read_sql_query(query_3, con)

 Total_Payload_Mass_KG
                 13048


### Task 4: Display average payload mass carried by booster version `F9 v1.1`

In [5]:
query_4 = """
SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_Mass_KG
FROM SPACEXTBL
WHERE Booster_Version = 'F9 v1.1';
"""
pd.read_sql_query(query_4, con)

 Avg_Payload_Mass_KG
              3325.0


### Task 5: List the date when the first successful landing outcome on a ground pad was achieved

In [6]:
query_5 = """
SELECT MIN(Date) AS First_Ground_Pad_Success
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (ground pad)';
"""
pd.read_sql_query(query_5, con)

First_Ground_Pad_Success
              2015-12-22


### Task 6: List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000

In [7]:
query_6 = """
SELECT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;
"""
pd.read_sql_query(query_6, con)

Booster_Version  PAYLOAD_MASS__KG_
    F9 FT B1022               4696
    F9 FT B1026               4600


### Task 7: List the total number of successful and failure mission outcomes

In [8]:
query_7 = """
SELECT Mission_Outcome, COUNT(*) AS Total_Count
FROM SPACEXTBL
GROUP BY Mission_Outcome;
"""
pd.read_sql_query(query_7, con)

    Mission_Outcome  Total_Count
Failure (in flight)            1
            Success           17


### Task 8: List the names of the booster versions which have carried the maximum payload mass

In [9]:
query_8 = """
SELECT DISTINCT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTBL
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL);
"""
pd.read_sql_query(query_8, con)

Booster_Version  PAYLOAD_MASS__KG_
  F9 B5 B1049.4              15600
  F9 B5 B1049.7              15600


### Task 9: List the records which will display the month names, failure landing outcomes in drone ship, booster versions, and launch sites for the months in year 2015

In [10]:
query_9 = """
SELECT substr(Date, 6, 2) AS Month,
       Landing_Outcome,
       Booster_Version,
       Launch_Site
FROM SPACEXTBL
WHERE Landing_Outcome = 'Failure (drone ship)'
  AND substr(Date, 1, 4) = '2015';
"""
pd.read_sql_query(query_9, con)

Month      Landing_Outcome Booster_Version Launch_Site
   01 Failure (drone ship)   F9 v1.1 B1012 CCAFS LC-40
   04 Failure (drone ship)   F9 v1.1 B1015 CCAFS LC-40


### Task 10: Rank the count of landing outcomes between `2010-06-04` and `2017-03-20` in descending order

In [11]:
query_10 = """
SELECT Landing_Outcome, COUNT(*) AS Outcome_Count
FROM SPACEXTBL
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Outcome_Count DESC;
"""
pd.read_sql_query(query_10, con)

       Landing_Outcome  Outcome_Count
            No attempt              4
  Success (drone ship)              3
  Success (ground pad)              2
   Failure (parachute)              2
  Failure (drone ship)              2
  Uncontrolled (ocean)              1
Precluded (drone ship)              1


In [12]:
# Close database connection
con.close()